In [1]:
from pathlib import Path
import sys

import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data import PatchDataset
from models import CustomVAE, DDPM, TimeUNet

DATA_DIR = ROOT / "data" / "images"
VAE_CKPT = ROOT / "microlad-anode" / "vae_anode.pth"
UNET_CKPT = ROOT / "microlad-anode" / "unet_anode.pth"
PATCH_SIZE = 64
BATCH_SIZE = 2
TIMESTEPS = 1000
SEED = 0


In [2]:
dataset = PatchDataset(DATA_DIR, patch_size=PATCH_SIZE, seed=SEED)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
batch = next(iter(loader))
batch.shape


torch.Size([2, 1, 64, 64])

In [3]:
vae = CustomVAE(latent_ch=4)
vae.load_state_dict(torch.load(VAE_CKPT, map_location="cpu")["vae"])
vae.eval()

unet = TimeUNet(latent_ch=4)
unet.load_state_dict(torch.load(UNET_CKPT, map_location="cpu"))
unet.eval()

ddpm = DDPM(timesteps=TIMESTEPS)


In [4]:
with torch.no_grad():
    z, _ = vae.encode(batch * 2 - 1)
    t = torch.randint(0, TIMESTEPS, (z.shape[0],))
    noise = torch.randn_like(z)
    z_t = ddpm.q_sample(z, t, noise)
    pred_noise = unet(z_t, t)
    z_prev = ddpm.p_sample(unet, z_t, t)

batch.shape, z.shape, z_t.shape, pred_noise.shape, z_prev.shape


(torch.Size([2, 1, 64, 64]),
 torch.Size([2, 4, 16, 16]),
 torch.Size([2, 4, 16, 16]),
 torch.Size([2, 4, 16, 16]),
 torch.Size([2, 4, 16, 16]))